In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 4 ###

def create_dataframe_of_counts_16(dataframe, column, rename_index, rename_column, return_percentages=False):
    # drop the first row using GPU-based iloc (no CPU __getitem__ here)
    df = dataframe.iloc[1:]
    # compute raw counts on GPU via groupby.size()
    vc = df.groupby(column).size()
    if return_percentages:
        # convert to percentages on GPU
        vc = vc / len(df) * 100
    # reset_index and rename entirely on GPU
    df_counts = vc.reset_index()
    return df_counts.rename(columns={
        column:     rename_index,
        'size':     rename_column
    })

# call the optimized function (no slicing outside)
percentages_per_country_df = create_dataframe_of_counts_16(
    responses_df_2022_cell10,
    'In which country do you currently reside?',
    'country',
    '% of respondents',
    return_percentages=False
)

# inspect the resulting GPU DataFrame
percentages_per_country_df.info()